# Session 1: where a 3D point lands, and why two cameras give depth
**A. Real-world problem.** Close one eye and hold a finger at arm's length, then switch eyes:
the finger jumps sideways against the background. The jump is large for near things and small
for far things. Engineers use the same effect, with two cameras, to measure the shape of a
loaded rubber strip or a bridge cable without touching it.

**Plan.** (1) write the pinhole camera rule, (2) put two cameras side by side, (3) find the
sideways jump (the *disparity*), (4) turn the jump back into depth, (5) see why far things are
hard to measure.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F = 1000.0      # focal length measured in pixels (how far the pinhole is from the sensor, in pixel widths)
BASE = 0.04     # distance between the two cameras in metres

## Step 1: the pinhole camera
Light from a point (X, Y, Z) passes through a tiny hole and lands on a sensor a distance f
behind it. By similar triangles, the landing place is

    u = f * X / Z        v = f * Y / Z

X is sideways, Y is up, Z is the distance in front of the camera, all in metres. Because f is
counted in pixels, u and v come out in pixels measured from the middle of the image.
Check by hand: f = 1000 px, a point at X = 0.02 m, Z = 0.5 m lands at u = 1000 * 0.02 / 0.5 = 40 px.

In [ ]:
def project(point, f=F):
    """Pixel coordinates (u, v), measured from the image centre, of a 3D point (X, Y, Z) in metres."""
    X, Y, Z = point
    if Z <= 0:
        raise ValueError("the point must be in front of the camera (Z > 0)")
    return f * X / Z, f * Y / Z

## Step 2: two cameras side by side
Put camera 2 a distance B (the *baseline*) to the right of camera 1, both looking straight ahead.
For camera 2 the point is B closer to the left, so its sideways distance is X - B:

    u1 = f * X / Z        u2 = f * (X - B) / Z

The sideways jump between the two pictures is the **disparity** d = u1 - u2 = f * B / Z.
It does not depend on X at all: only on the depth Z. That is why two views give depth.

In [ ]:
def stereo_project(point, f=F, baseline=BASE):
    """Return ((u1, v), (u2, v)) for camera 1 at x = 0 and camera 2 at x = baseline."""
    X, Y, Z = point
    return project((X, Y, Z), f), project((X - baseline, Y, Z), f)


def disparity(point, f=F, baseline=BASE):
    (u1, _), (u2, _) = stereo_project(point, f, baseline)
    return u1 - u2

## Step 3: from disparity back to depth
Turn d = f B / Z around: **Z = f B / d**. Measure the jump in pixels, and the depth follows.
Example: f = 1000 px, B = 0.04 m, d = 80 px gives Z = 1000 * 0.04 / 80 = 0.5 m.

In [ ]:
def depth_from_disparity(d, f=F, baseline=BASE):
    """Depth in metres from a disparity in pixels. A disparity of zero or less means 'infinitely far' (returns inf)."""
    d = np.asarray(d, dtype=float)
    with np.errstate(divide="ignore"):
        z = np.where(d > 0, f * baseline / np.where(d > 0, d, 1.0), np.inf)
    return float(z) if z.ndim == 0 else z

## Step 4: why far things are hard
Disparity is measured in whole pixels at best (Session 2 does better). Neighbouring disparities
d and d + 1 pixel correspond to depths f B / d and f B / (d + 1). The gap between them is the
depth step the cameras cannot resolve. It grows rapidly with distance.

In [ ]:
def depth_step(Z, f=F, baseline=BASE):
    """Depth change in metres caused by a 1 pixel change of disparity, for a point at depth Z."""
    d = f * baseline / Z
    return f * baseline / d - f * baseline / (d + 1.0)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    p = (0.02, 0.01, 0.5)
    (u1, v1), (u2, v2) = stereo_project(p)
    print(f"point {p}: camera 1 sees ({u1:.1f}, {v1:.1f}) px, camera 2 sees ({u2:.1f}, {v2:.1f}) px")
    print(f"disparity {disparity(p):.1f} px  ->  depth {depth_from_disparity(disparity(p)):.3f} m")
    for Z in (0.3, 0.5, 1.0, 2.0, 5.0):
        print(f"Z = {Z:4.1f} m: disparity {F * BASE / Z:6.1f} px, one pixel of disparity = {1000 * depth_step(Z):8.2f} mm of depth")

    # top view: two cameras, three points at different depths
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    for x, name in ((0.0, "camera 1"), (BASE, "camera 2")):
        ax.plot([x], [0], "ks", ms=8)
        ax.annotate(name, (x, 0), textcoords="offset points", xytext=(-6 if x == 0 else 6, -16), ha="right" if x == 0 else "left", fontsize=8)
    for Z, col in ((0.3, "tab:red"), (0.6, "tab:orange"), (1.2, "tab:green")):
        X = 0.02
        ax.plot([X], [Z], "o", color=col)
        ax.plot([0, X], [0, Z], "-", color=col, lw=0.8)
        ax.plot([BASE, X], [0, Z], "--", color=col, lw=0.8)
        ax.annotate(f"Z = {Z} m, disparity {F * BASE / Z:.0f} px", (X, Z), textcoords="offset points", xytext=(8, 0), fontsize=8)
    ax.set_xlabel("sideways position X (m)")
    ax.set_ylabel("depth Z (m)")
    ax.set_title("Top view: nearer points are seen at more different angles")
    ax.set_xlim(-0.04, 0.16)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "top_view.png", dpi=120)

    Z = np.linspace(0.25, 3.0, 200)
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(Z, F * BASE / Z, label="d = f B / Z  (f = 1000 px, B = 0.04 m)")
    ax2.plot([0.5, 1.0, 2.0], [F * BASE / z for z in (0.5, 1.0, 2.0)], "ro")
    ax2.set_xlabel("depth Z (m)")
    ax2.set_ylabel("disparity d (pixels)")
    ax2.set_title("Disparity falls as 1/Z: depth is easier to measure up close")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "disparity_vs_depth.png", dpi=120)

## Where this stops
We assumed two identical cameras, perfectly parallel, with known f and B. Session 2 handles
real pixels, sub-pixel matching and the accuracy of the result.

## Coding tasks
1. **Easy.** Print the disparity of a point at 0.25 m, 0.5 m and 1 m with `disparity((0, 0, Z))` and predict the three values before you run it.
2. **Stretch.** Write `baseline_for_step(Z, step)` that returns the baseline for which one pixel of disparity is `step` metres of depth at depth Z, and check it against Assessment D.
3. **Explain.** Explain, using the top-view figure, why a point at infinity has disparity 0.

## Assessment
- **A. Conceptual.** Why does the disparity not depend on the sideways position X of the point?
- **B. Numerical.** Two cameras 0.06 m apart with f = 1000 px measure a disparity of 25 px. How far away is the point? Is the answer sensible?
- **C. Graph and simulation interpretation.** In the graph of disparity against depth, what happens to the disparity when the depth is doubled from 0.5 m to 1.0 m, and what does that mean for the accuracy?
- **D. Design and optimisation challenge.** Design: with f = 1000 px, choose a baseline so that one pixel of disparity corresponds to at most 5 mm of depth at 0.5 m. What is the cost of a longer baseline?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    u, v = L.project((0.02, 0.01, 0.5))
    checks.append(("hand calculation: u = 1000 * 0.02 / 0.5 = 40 px, v = 20 px", abs(u - 40) < 1e-9 and abs(v - 20) < 1e-9))
    checks.append(("doubling the depth halves the image coordinates", abs(L.project((0.02, 0.0, 1.0))[0] - u / 2) < 1e-9))
    d = L.disparity((0.02, 0.01, 0.5))
    checks.append((f"worked example: disparity f B / Z = 1000 * 0.04 / 0.5 = 80 px (got {d:.1f})", abs(d - 80.0) < 1e-9))
    checks.append(("worked example: depth from d = 80 px is 0.5 m", abs(L.depth_from_disparity(80.0) - 0.5) < 1e-12))
    checks.append(("disparity does not depend on the sideways position X",
                   abs(L.disparity((-0.1, 0.0, 0.8)) - L.disparity((0.3, 0.2, 0.8))) < 1e-9))
    zs = np.array([0.3, 0.7, 1.5, 4.0])
    ds = np.array([L.disparity((0.01, 0.0, z)) for z in zs])
    checks.append(("round trip: depth_from_disparity(disparity(Z)) = Z", np.allclose(L.depth_from_disparity(ds), zs)))
    checks.append(("a nearer point has a larger disparity", bool(np.all(np.diff(ds) < 0))))
    checks.append(("edge: disparity 0 means infinitely far", L.depth_from_disparity(0.0) == np.inf and L.depth_from_disparity(-3.0) == np.inf))
    try:
        L.project((0.0, 0.0, 0.0))
        behind = False
    except ValueError:
        behind = True
    checks.append(("edge: a point at Z = 0 is rejected", behind))
    checks.append(("one pixel of disparity at 0.5 m is about 6.2 mm of depth", abs(1000 * L.depth_step(0.5) - 6.17) < 0.05))
    checks.append(("depth step grows with distance", L.depth_step(2.0) > 10 * L.depth_step(0.5)))
    checks.append(("design answer: baseline 0.05 m gives a depth step of 4.95 mm at 0.5 m, 0.04 m gives 6.17 mm",
                   abs(1000 * L.depth_step(0.5, baseline=0.05) - 4.95) < 0.01 and abs(1000 * L.depth_step(0.5) - 6.17) < 0.01))
    checks.append(("hand calculation: f = 1000, B = 0.06, d = 25 px gives Z = 2.4 m", abs(L.depth_from_disparity(25.0, 1000.0, 0.06) - 2.4) < 1e-12))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
